# 01 — Getting Started mit Claude Tool Use

Dieses Notebook zeigt, wie man eigene Tools für Claude definiert und nutzt.

## Setup

In [ ]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
print("Client bereit.")

## Step 0 — Tool-Funktionalität schreiben

Bevor Claude ein Tool nutzen kann, muss die Funktion selbst implementiert werden.

Beispiel: `get_stock_price` erwartet einen Firmennamen oder Ticker und gibt den aktuellen Kurs zurück.

In [ ]:
def get_stock_price(company: str) -> dict:
    """
    Sendet eine Anfrage an eine Börsen-API und gibt
    den aktuellen Aktienkurs für das angegebene Unternehmen zurück.

    Returns:
        dict mit 'symbol' und 'price'
    """
    # Platzhalter — hier echte API-Anbindung einfügen
    mock_data = {
        "General Motors": {"symbol": "GM",   "price": 43.09},
        "Apple":          {"symbol": "AAPL",  "price": 189.50},
        "Tesla":          {"symbol": "TSLA",  "price": 245.00},
    }
    return mock_data.get(company, {"symbol": "UNKNOWN", "price": None})


# Test
print(get_stock_price("General Motors"))

## Step 1 — Tool definieren und API-Request senden

Claude muss wissen, welche Tools existieren: Name, Beschreibung und Input-Schema.

In [ ]:
tool_definition = {
    "name": "get_stock_price",
    "description": "Retrieves the current stock price for a given company",
    "input_schema": {
        "type": "object",
        "properties": {
            "company": {
                "type": "string",
                "description": "The company name to fetch stock data for"
            }
        },
        "required": ["company"]
    }
}

response = client.messages.create(
    model="claude-opus-4-6",
    messages=[{"role": "user", "content": "How many shares of General Motors can I buy with $500?"}],
    max_tokens=500,
    tools=[tool_definition]
)

print("stop_reason:", response.stop_reason)
print("content:", response.content)

## Step 2 — Claude antwortet mit Tool-Use-Request

Claude erkennt, dass `get_stock_price` benötigt wird, und antwortet mit `stop_reason: "tool_use"`.

In [ ]:
# Tool-Use-Block aus der Antwort extrahieren
tool_use_block = next(block for block in response.content if block.type == "tool_use")

tool_name  = tool_use_block.name
tool_input = tool_use_block.input
tool_id    = tool_use_block.id

print("Tool:", tool_name)
print("Input:", tool_input)

## Step 3 — Tool ausführen und Ergebnis an Claude zurückgeben

Wir rufen die echte Funktion auf und schicken das Ergebnis als `tool_result` zurück.

In [ ]:
import json

# Funktion mit Claudes Input aufrufen
tool_result = get_stock_price(tool_input["company"])
print("Tool-Ergebnis:", tool_result)

# Ergebnis als tool_result zurück an Claude senden
final_response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=500,
    tools=[tool_definition],
    messages=[
        {"role": "user",      "content": "How many shares of General Motors can I buy with $500?"},
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_id,
                    "content": json.dumps(tool_result)
                }
            ]
        }
    ]
)

print("\nClaudes Antwort:")
print(final_response.content[0].text)

## Step 4 — Claude formuliert die finale Antwort

Claude erhält das Tool-Ergebnis und antwortet auf die ursprüngliche Nutzerfrage — z.B.:

> *"With a current share price of $43.09, you can buy approximately 11 shares of General Motors with $500."*

Die komplette Tool-Use-Schleife ist damit abgeschlossen:

```
User Prompt
    → Claude erkennt Tool-Bedarf (stop_reason: tool_use)
        → Client führt Tool aus
            → Ergebnis als tool_result zurück an Claude
                → Claude antwortet mit finaler Antwort
```

---

## Quiz

### Frage 1

Welche der folgenden Anwendungsfälle eignen sich gut für Tool Use?

- **(a)** Ein Kundensupport-Chatbot, der Bestellinfos aus einer Datenbank abrufen muss.
- **(b)** Ein Schreibassistent, der Grammatik und Stil verbessert.
- **(c)** Ein Finanzberater, der personalisierte Anlageempfehlungen gibt.
- **(d)** Ein Reise-Chatbot, der Flüge und Hotels bucht.

<details>
<summary>Antwort anzeigen</summary>

**a, c und d**

- **(a)** Externes Datenbankabfragen → Tool Use sinnvoll
- **(c)** Berechnungen und externe Daten → Tool Use sinnvoll
- **(d)** Interaktion mit externen Systemen → Tool Use sinnvoll

**(b)** kann durch Claudes eingebaute Sprachfähigkeiten abgedeckt werden — kein Tool Use nötig.

</details>

---

### Frage 2

Bringe die folgenden Schritte des Tool-Use-Ablaufs in die richtige Reihenfolge:

- **(a)** Claude nutzt das Tool-Ergebnis für die finale Antwort.
- **(b)** Client-Code extrahiert Tool-Namen und Input aus Claudes Anfrage.
- **(c)** Claude bewertet den Prompt und gibt einen Tool-Use-Request aus.
- **(d)** Client-Code führt das Tool aus und gibt das Ergebnis zurück.
- **(e)** Client-Code stellt Claude die verfügbaren Tools und den User-Prompt bereit.

<details>
<summary>Antwort anzeigen</summary>

**Richtige Reihenfolge: e → c → b → d → a**

1. **(e)** Tools und Prompt an Claude übergeben
2. **(c)** Claude entscheidet sich für ein Tool
3. **(b)** Client extrahiert Tool-Name und Input
4. **(d)** Client führt das Tool aus und gibt Ergebnis zurück
5. **(a)** Claude formuliert die finale Antwort

</details>